<a href="https://colab.research.google.com/github/Maxxx-VS/IMA_SibADI/blob/main/ML_2_3_%D0%9E%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%B8%D0%B5_%D0%BD%D0%B5%D0%B9%D1%80%D0%BE%D1%81%D0%B5%D1%82%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Задание 2.3. Обучение нейросети с 2 входами (нелинейная активация нейрона)

In [1]:
!pip install -q onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 22.2 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv('/content/multiregress-092022.csv')
df = df.drop('i', axis=1)
x = df[['Me', 'ne']].to_numpy()
y = df.tc.to_numpy()
n = y.size

In [3]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler = StandardScaler(with_mean=True, with_std=True)
x_s = scaler.fit_transform(x)
y = y.reshape((-1, 1))
scaler2 = MinMaxScaler(feature_range=(-1, 1))
y_s = scaler2.fit_transform(y)

In [4]:
np.random.seed(42)
b = np.random.rand(1)
w = np.random.randn(2, 1)

In [5]:
n_epochs = 500
losses = []

lr = 0.5
for epoch in range(n_epochs):
    a = b + x_s @ w
    yhat = np.tanh(a)
    error = yhat - y_s
    loss = np.mean(error**2)
    losses.append(loss)
    b_grad = 2 * (error * (1 - yhat**2)).mean()
    w_grad = 2 * (x_s.T @ (error * (1 - yhat**2))) / n
    if epoch % 50 == 0:
        print('Epoch: {}, b_grad: {:.4f}, w_grad: {}, b: {:.4f}, w: {}'.format(
            epoch, float(b_grad), w_grad.ravel(), float(b[0]), w.ravel()))
    b -= lr * b_grad
    w -= lr * w_grad
print('Обучение закончено: ', ' b: ', b, ' w: ', w)

Epoch: 0, b_grad: -0.0429, w_grad: [-0.59248775 -0.08369967], b: 0.3745, w: [-1.11188012  0.31890218]
Epoch: 50, b_grad: 0.0000, w_grad: [-1.13618899e-07 -5.73619366e-08], b: -0.1401, w: [0.66024614 0.20880518]
Epoch: 100, b_grad: 0.0000, w_grad: [-8.37410949e-14 -4.22603871e-14], b: -0.1401, w: [0.66024638 0.2088053 ]
Epoch: 150, b_grad: 0.0000, w_grad: [-6.24500451e-17 -5.67727683e-18], b: -0.1401, w: [0.66024638 0.2088053 ]
Epoch: 200, b_grad: 0.0000, w_grad: [-6.24500451e-17 -5.67727683e-18], b: -0.1401, w: [0.66024638 0.2088053 ]
Epoch: 250, b_grad: 0.0000, w_grad: [-6.24500451e-17 -5.67727683e-18], b: -0.1401, w: [0.66024638 0.2088053 ]
Epoch: 300, b_grad: 0.0000, w_grad: [-6.24500451e-17 -5.67727683e-18], b: -0.1401, w: [0.66024638 0.2088053 ]
Epoch: 350, b_grad: 0.0000, w_grad: [-6.24500451e-17 -5.67727683e-18], b: -0.1401, w: [0.66024638 0.2088053 ]
Epoch: 400, b_grad: 0.0000, w_grad: [-6.24500451e-17 -5.67727683e-18], b: -0.1401, w: [0.66024638 0.2088053 ]
Epoch: 450, b_grad:

In [6]:
import plotly.graph_objects as go
fig3 = go.Figure()
fig3.add_trace(go.Scatter(y=losses,
                          mode='markers', name='loss',
                          marker=dict(color='black', size=7), opacity=0.8))
fig3.update_layout(title_text="MSE vs epoch", title_font_size=20,
                   xaxis_title="epoch", yaxis_title="MSE")
fig3.show()

In [7]:
import torch
from torch import nn

class NeuronTanh(nn.Module):
    def __init__(self):
        super(NeuronTanh, self).__init__()
        self.a = nn.Linear(bias=True, in_features=2, out_features=1)
        self.f = nn.Tanh()

    def forward(self, x):
        x = self.f(self.a(x))
        return x

In [8]:
model = NeuronTanh()

In [9]:
x_torch = torch.as_tensor(x_s).float()
y_torch = torch.as_tensor(y_s).float()

In [10]:
from torch import optim
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.5)

In [11]:

n_epochs = 500
losses = []

for epoch in range(n_epochs):
    loss = 0.0
    model.train()
    optimizer.zero_grad()
    y_hat_torch = model(x_torch)
    loss = loss_fn(y_hat_torch, y_torch)
    loss.backward()
    optimizer.step()
    losses.append(loss.detach().numpy())
    if epoch % 50 == 0:
        print('Epoch: {}, Loss: {:.2f}'.format(epoch, loss))

Epoch: 0, Loss: 0.18
Epoch: 50, Loss: 0.02
Epoch: 100, Loss: 0.02
Epoch: 150, Loss: 0.02
Epoch: 200, Loss: 0.02
Epoch: 250, Loss: 0.02
Epoch: 300, Loss: 0.02
Epoch: 350, Loss: 0.02
Epoch: 400, Loss: 0.02
Epoch: 450, Loss: 0.02


In [12]:
print(model.a.bias, model.a.weight)

Parameter containing:
tensor([-0.1401], requires_grad=True) Parameter containing:
tensor([[0.6602, 0.2088]], requires_grad=True)


In [13]:
fig3.add_trace(go.Scatter(y=losses,
                          mode='markers+lines', name='PyTorch loss',
                          marker=dict(color='red', size=3), opacity=0.8))
fig3.show()

In [14]:
import warnings

model.eval()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    _ = torch.onnx.export(
        model, x_torch[:1], "neuron_two_inputs_tanh.onnx",
        input_names=['x'], output_names=['y'])
print("Модель сохранена: neuron_two_inputs_tanh.onnx")

[torch.onnx] Obtain model graph for `NeuronTanh([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `NeuronTanh([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Модель сохранена: neuron_two_inputs_tanh.onnx
